# 08. 화재 활동 지수: 정의와 검증

달력에는 탐지 수 대신 **화재 활동 지수**를 보여 줍니다. 그러면 보는 사람이 "MODIS 몇 건"과 "VIIRS 몇 건"을 비교할 일이 없습니다.

| 단계 | 내용 (`fsml/activity.py`) |
| --- | --- |
| 1. 합계 | 지역(10° 블록, 사용자가 그린 영역 등) × 월로 조화된 탐지 수를 더함. 불 없는 달은 0 |
| 2. 백분위 | 같은 지역, 같은 달의 평년(2003~2023년) 21개 값 중 위치 (0~100, 같은 값은 절반으로 셈) |
| 3. 단계 | 매우 낮음 <10 ≤ 낮음 <30 ≤ 보통 <70 ≤ 높음 <90 ≤ **매우 높음(이상)** |

**검증 질문:** MODIS가 꺼진 뒤에는 VIIRS 환산값으로 지수를 만듭니다. 그 지수가 실제 MODIS로 만든 지수와 얼마나 같은가?
2020~2024년을 한 해씩 가리고(그 전 해들로 학습한 모델의 예측만 사용), 세 가지 방법을 실제 MODIS 지수와 비교합니다.

| 방법 | 설명 |
| --- | --- |
| AI 환산 (LightGBM) | 채택한 모델의 환산값을 MODIS 평년과 비교 |
| 단순 비율 환산 | 위도대 × 월 비율 하나로 환산 (기준선) |
| VIIRS 자체 백분위 | 환산 없이 VIIRS를 VIIRS 평년(2012~2023년)과 비교. "센서별 지표를 이어 붙이기" 방식 |

평가는 평년에 한 번이라도 불이 난 지역-월만 합니다(늘 0인 곳은 누구나 "보통"을 맞힘).

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

from fsml import evaluate, gbm  # evaluate: 한글 폰트 설정 포함
from fsml.activity import BASELINE_YEARS, LEVEL_EDGES, LEVELS, activity_index, agreement, monthly_totals, percentile
from fsml.baseline import RatioBaseline
from fsml.export import AQUA_START
from fsml.features import MODIS_RELIABLE_UNTIL, make_features, training_rows
from fsml.paths import EXPORTS, default_grid
from fsml.splits import auto_folds, block_id

GRID = default_grid()
grid = pd.read_parquet(GRID)
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(training_rows(grid), res)
kind, folds = auto_folds(data, n=5)
assert kind == "time", "이 노트북은 여러 해 실제 격자(grid_0.5_month.parquet)가 있어야 합니다. 7일 샘플로는 돌리지 않습니다."
year = pd.to_datetime(data["period"]).dt.year
TEST_YEARS = [int(year[te].iloc[0]) for _, _, te in folds]
print(f"{GRID.name} | 시험 연도: {TEST_YEARS}")

## 시험 연도 예측 (각 해는 그 전 해들로만 학습)

In [ ]:
data["ai"] = np.nan
data["ratio"] = np.nan
t0 = time.time()
for tr, va, te in folds:
    model = gbm.fit(data, FEATURES, tr, va, log_every=0)
    data.loc[te, "ai"] = model.predict(data.loc[te, FEATURES])
    data.loc[te, "ratio"] = RatioBaseline().fit(data[tr | va]).predict(data[te])
print(f"완료 ({time.time() - t0:.0f}초)")

## 지수 비교

In [ ]:
METHODS = {"AI 환산 (LightGBM)": "ai_pct", "단순 비율 환산": "ratio_pct", "VIIRS 자체 백분위": "viirs_pct"}
period_all = pd.to_datetime(grid["period"])
truth_rows = grid[grid["modis_n"].notna() & period_all.between(AQUA_START, MODIS_RELIABLE_UNTIL)]
test_rows = data[data["ai"].notna()]


def index_table(deg):
    with_region = lambda d: d.assign(region=block_id(d, deg))
    t = monthly_totals(with_region(truth_rows), ["modis_n"], "region")
    for part, cols in ((with_region(test_rows), ["ai", "ratio"]), (with_region(data), ["viirs_n"])):
        p = monthly_totals(part, cols, "region")
        t = t.merge(p, on=["region", "period"], how="left")
        inside = t["period"].between(p["period"].min(), p["period"].max())
        t.loc[inside, cols] = t.loc[inside, cols].fillna(0)  # 그 기간에 행이 없는 지역 = 탐지 0
    t["modis_pct"] = percentile(t, "modis_n", "region")
    t["ai_pct"] = percentile(t, "ai", "region", base_col="modis_n")
    t["ratio_pct"] = percentile(t, "ratio", "region", base_col="modis_n")
    t["viirs_pct"] = percentile(t, "viirs_n", "region", years=(2012, 2023))

    month = t["period"].dt.month
    base = t[t["period"].dt.year.between(*BASELINE_YEARS)]
    ever = base.groupby(["region", base["period"].dt.month])["modis_n"].max() > 0
    active = pd.MultiIndex.from_arrays([t["region"], month]).map(ever).fillna(False).to_numpy(bool)
    ev = active & t["period"].dt.year.isin(TEST_YEARS).to_numpy() & t[list(METHODS.values())].notna().all(axis=1).to_numpy()
    return t, ev


tables, summary = {}, {}
for deg in (10, 2):
    t, ev = index_table(deg)
    tables[deg] = (t, ev)
    for name, col in METHODS.items():
        summary[(f"{deg}° 지역", name)] = agreement(t.loc[ev, "modis_pct"], t.loc[ev, col])
summary = pd.DataFrame(summary).T
summary.round(3)

**읽는 법:** "단계 일치"는 5단계가 정확히 같은 비율, "이상 F1"은 실제 MODIS가 "매우 높음"인 달을 얼마나 똑같이 잡는지입니다.
2° 지역은 사용자가 그리는 작은 관심 영역에 가깝습니다. 지역이 작을수록 탐지 수가 적어 지수가 더 흔들립니다.

In [ ]:
t, ev = tables[10]
yr = t["period"].dt.year
by_year = pd.DataFrame({
    name: {y: agreement(t.loc[ev & (yr == y).to_numpy(), "modis_pct"], t.loc[ev & (yr == y).to_numpy(), col])["이상 F1"]
           for y in TEST_YEARS}
    for name, col in METHODS.items()
})
ax = by_year.plot.bar(figsize=(9, 4), rot=0, color=["#2a78d6", "#eb6834", "#1baf7a"], edgecolor="#fcfcfb", linewidth=2)
ax.set_ylim(0, 1)
ax.set_xlabel("시험 연도")
ax.set_ylabel("이상 F1 (높을수록 좋음)")
ax.set_title("10° 지역: 실제 MODIS의 '매우 높음' 달을 얼마나 똑같이 잡나")
ax.grid(axis="y", color="#e4e3df", lw=0.8)
ax.set_axisbelow(True)
by_year.round(3)

## 달력 예시

`03_model`이 만든 `exports/harmonized.parquet`으로 한 지역의 25년 달력을 그립니다. 관심 영역의 칸을 더해 지수를 만드는 방식은
웹앱에서 사용자가 영역을 그릴 때와 같습니다. 오른쪽 표시는 그 해에 실제 MODIS 대신 쓴 값입니다.

In [ ]:
LEVEL_COLORS = ["#184f95", "#86b6ef", "#dcdbd6", "#f19b9a", "#b8302f"]  # 낮음(파랑) · 보통(회색) · 높음(빨강)
AOI = {"name": "아마존 남부 (브라질 마투그로수·파라)", "lat": (-15, -5), "lon": (-60, -50)}

h = pd.read_parquet(EXPORTS / "harmonized.parquet")
box = h[h["lat"].between(*AOI["lat"]) & h["lon"].between(*AOI["lon"])].assign(region=AOI["name"])
idx = activity_index(box, "harmonized", "region")
cal = idx.assign(year=idx["period"].dt.year, month=idx["period"].dt.month).pivot(index="year", columns="month", values="percentile")
source = box.assign(year=pd.to_datetime(box["period"]).dt.year).groupby("year")["source"].agg(lambda s: s.mode().iat[0])

fig, ax = plt.subplots(figsize=(8, 9))
mesh = ax.pcolormesh(cal.columns, cal.index, cal.to_numpy(), cmap=ListedColormap(LEVEL_COLORS),
                     norm=BoundaryNorm([0, *LEVEL_EDGES, 100.01], len(LEVELS)), edgecolors="#fcfcfb", linewidth=1.5)
ax.invert_yaxis()
ax.set_xticks(range(1, 13), [f"{m}월" for m in range(1, 13)])
ax.set_yticks(cal.index)
ax.set_title(f"화재 활동 지수: {AOI['name']}")
labels = {"terra_scaled": "Terra 보정", "viirs_model": "VIIRS 환산"}
for y, s in source.items():
    if s in labels and y in cal.index:
        ax.text(12.7, y, labels[s], va="center", fontsize=8, color="#52514e")
cbar = fig.colorbar(mesh, ax=ax, ticks=[5, 20, 50, 80, 95], fraction=0.04, pad=0.12)
cbar.ax.set_yticklabels(LEVELS)
plt.tight_layout()

In [ ]:
# 이상(매우 높음) 달 목록과 평년값 대비 배수
hot = idx[idx["percentile"] >= 90].assign(평년_대비=lambda d: d["harmonized"] / d["normal"].where(d["normal"] > 0))
hot[["period", "harmonized", "normal", "평년_대비", "percentile"]].round(1).tail(15)

## 메모

- 평년에 늘 0인 달은 백분위 50("보통")이 됩니다. 웹앱에서는 평년값(`normal`)이 0인 달을 흐리게 표시하는 게 좋습니다.
- 평년(2003~2023년)과 시험 연도(2020~2024년)가 일부 겹칩니다. 세 방법 모두 같은 조건이라 비교에는 영향이 없습니다.
- 다경 님의 `anomaly_agreement`(평균 + z × 표준편차 기준)는 이 지수(백분위 90 기준)와 이상의 정의가 다릅니다. 함께 쓰려면 정의를 맞춥니다.